# Modelo base: regresión logística

* **Objetivo:** $y_{m,t} = 1$ si el municipio $m$ está en brote en la semana $t+4$ (canal endémico).
* **Modelo base:** regresión logística con penalización L2, dentro de un `Pipeline` (imputación → estandarización → modelo), ajustada con `GridSearchCV` y **validación cruzada hacia adelante por años con purga** (el mismo tipo de partición que entrenamiento/prueba).
* **Líneas base triviales:** `DummyClassifier` (proporción de entrenamiento), **persistencia** (¿hay brote ahora?), **persistencia de 8 semanas** (proporción reciente de semanas en brote) e **ingenuo estacional** (¿hubo brote en la misma semana hace un año?).
* **Selección:** AUC-PR en la validación cruzada (clase minoritaria); umbral de decisión que maximiza F1 en la validación de 2022. La prueba 2023–2024 se usa **una sola vez**, en este capítulo.

In [1]:
import os  # Variables de entorno y rutas
import sys  # Configuración del intérprete
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count()))  # Evita un aviso de joblib en Windows
sys.path.insert(0, os.path.abspath("../.."))  # Raíz del proyecto (paquete src)
sys.path.insert(0, os.path.abspath(".."))  # Carpeta entregable1 (paquete e1)
import warnings  # Control de avisos
warnings.filterwarnings("ignore")  # Oculta avisos de versiones que no afectan los resultados
import contextlib, io  # Silenciar una salida de diagnóstico
from joblib.externals.loky import cpu_count  # Conteo de núcleos de joblib
with contextlib.redirect_stderr(io.StringIO()):  # En Windows 11 joblib imprime un diagnóstico al no encontrar «wmic»
    cpu_count(only_physical_cores=True)  # Se calcula una vez (queda en caché) sin ensuciar las salidas
import numpy as np  # Cálculo numérico
import pandas as pd  # Tablas
import plotly.express as px  # Figuras rápidas
import plotly.graph_objects as go  # Figuras detalladas
from plotly.subplots import make_subplots  # Figuras con varios paneles
from src import config as cf  # Configuración del proyecto (horizonte, periodos, variables)
from e1 import datos as d, estadistica as es, espacial as sp, modelo_base as mb  # Código del entregable
from e1.figuras import mostrar, mapa_base, AZUL, ROJO, VERDE, AMBAR, GRIS, MORADO, PERIODO_COLOR, CUADRANTE_COLOR  # Estilo
np.random.seed(cf.SEMILLA)  # Semilla global (42)
pd.set_option("display.width", 160)  # Ancho de las tablas
pd.set_option("display.max_columns", 40)  # Columnas visibles
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")  # Cuatro decimales
from sklearn.metrics import average_precision_score, confusion_matrix, precision_recall_curve, roc_auc_score, roc_curve  # Métricas y curvas
df = d.panel()  # Panel completo
mod = d.modelado(df)  # Filas aptas para modelar
tr, va, te = d.particion(mod)  # Particiones purgadas
print(f"Entrenamiento {len(tr):,} · validación {len(va):,} · prueba {len(te):,} filas")  # Tamaños

Entrenamiento 132,728 · validación 18,048 · prueba 37,600 filas


## Entrenamiento: Pipeline + GridSearchCV con validación purgada

In [2]:
grid = mb.ajustar_base(tr)  # Prueba 10 combinaciones × 4 pliegues
res = pd.DataFrame(grid.cv_results_)  # Resultados de la validación cruzada
tabla = pd.DataFrame({"C": res["param_clf__C"], "class_weight": res["param_clf__class_weight"].astype(str),  # Hiperparámetros
                      "AUC-PR (media)": res["mean_test_auc_pr"], "AUC-PR (desv.)": res["std_test_auc_pr"],  # Criterio
                      "AUC-ROC (media)": res["mean_test_auc_roc"], "AUC-ROC entrenamiento": res["mean_train_auc_roc"],  # Diagnóstico de sobreajuste
                      "Brier (media)": -res["mean_test_neg_brier"]}).sort_values("AUC-PR (media)", ascending=False)  # Calibración
print("Mejores hiperparámetros:", grid.best_params_)  # Elección
tabla.round(4)

Mejores hiperparámetros: {'clf__C': 0.01, 'clf__class_weight': None}


,C,class_weight,AUC-PR (media),AUC-PR (desv.),AUC-ROC (media),AUC-ROC entrenamiento,Brier (media)
2,0.0100,None,0.6433,0.0821,0.8985,0.9011,0.0631
4,0.1000,None,0.6429,0.0819,0.8980,0.9009,0.0630
6,1.0000,None,0.6422,0.0820,0.8973,0.9007,0.0630
8,10.0000,None,0.6419,0.0821,0.8970,0.9007,0.0630
0,0.0010,None,0.6414,0.0826,0.8998,0.9016,0.0635
3,0.0100,balanced,0.6373,0.0828,0.9015,0.9040,0.0969
5,0.1000,balanced,0.6370,0.0824,0.9010,0.9040,0.0969
7,1.0000,balanced,0.6366,0.0825,0.9005,0.9040,0.0970
9,10.0000,balanced,0.6365,0.0826,0.9004,0.9040,0.0971
1,0.0010,balanced,0.6364,0.0840,0.9020,0.9036,0.0979


In [3]:
mejor = grid.best_index_  # Fila de la mejor combinación
pliegues = pd.DataFrame({"año validado": [2018, 2019, 2020, 2021],  # Desempeño por pliegue
                         "AUC-PR": [res.loc[mejor, f"split{i}_test_auc_pr"] for i in range(4)],
                         "AUC-ROC": [res.loc[mejor, f"split{i}_test_auc_roc"] for i in range(4)]}).set_index("año validado")
pliegues.round(4)

,AUC-PR,AUC-ROC
año validado,,
2018,0.6089,0.8969
2019,0.7812,0.8956
2020,0.6184,0.8993
2021,0.5648,0.9022


**Interpretación.** El AUC-PR cambia muy poco entre valores de $C$: con 130 000 filas la regularización importa poco para el desempeño, pero la más fuerte razonable ($C = 0{,}01$) estabiliza los coeficientes ante la multicolinealidad. `class_weight="balanced"` sube levemente el AUC-ROC pero baja el AUC-PR y empeora la calibración (infla las probabilidades), así que no se pondera. El desempeño por pliegue varía con el año (2019, año epidémico, es el más fácil en AUC-PR), lo que confirma que una sola partición no basta para estimar la incertidumbre.

## Umbral de decisión (validación 2022) y evaluación en la prueba 2023–2024

In [4]:
modelo = grid.best_estimator_  # Pipeline reentrenado con todo el entrenamiento
p_va = modelo.predict_proba(va[cf.FEATURES])[:, 1]  # Probabilidades en validación
p_te = modelo.predict_proba(te[cf.FEATURES])[:, 1]  # Probabilidades en prueba (única vez)
umbral = mb.umbral_f1(va[cf.TARGET], p_va)  # Umbral que maximiza F1 en 2022
lb_va, lb_te = mb.lineas_base(tr, va), mb.lineas_base(tr, te)  # Líneas base
umbrales = {"Regresión logística": umbral, "Dummy (prior)": 0.5, "Persistencia": 1.0,  # Reglas binarias: alerta si el indicador es 1
            "Persistencia (8 sem.)": mb.umbral_f1(va[cf.TARGET], lb_va["Persistencia (8 sem.)"]), "Ingenuo estacional": 1.0}  # Persistencia continua: F1 en 2022
puntajes = {"Regresión logística": p_te, **lb_te}  # Puntajes en prueba
print(f"Umbral de la regresión logística (máximo F1 en 2022): {umbral:.3f}")  # Umbral
print(f"Prevalencia en la prueba: {te[cf.TARGET].mean():.1%} (entrenamiento {tr[cf.TARGET].mean():.1%})")  # Cambio de prevalencia

Umbral de la regresión logística (máximo F1 en 2022): 0.244
Prevalencia en la prueba: 30.3% (entrenamiento 10.3%)


In [5]:
resultados = {}  # Métricas con IC 95 % (bootstrap de 1 000 réplicas por municipio)
for nombre, p in puntajes.items():  # Cada modelo
    resultados[nombre] = mb.bootstrap_metricas(te[cf.TARGET], p, umbrales[nombre], te["COD_MUN_N"], B=1000)  # Métricas e IC
tab = pd.concat({n: r.apply(lambda f: f"{f['valor']:.3f} [{f['IC 2,5 %']:.3f}; {f['IC 97,5 %']:.3f}]", axis=1) for n, r in resultados.items()}, axis=1).T  # Formato valor [IC]
tab

,AUC-ROC,AUC-PR,Accuracy,Precision,Recall,F1,MCC,Brier
Regresión logística,0.885 [0.875; 0.895],0.809 [0.782; 0.833],0.825 [0.814; 0.836],0.700 [0.677; 0.720],0.742 [0.718; 0.765],0.720 [0.698; 0.741],0.594 [0.568; 0.616],0.122 [0.115; 0.129]
Dummy (prior),0.500 [0.500; 0.500],0.303 [0.281; 0.327],0.697 [0.673; 0.719],0.000 [0.000; 0.000],0.000 [0.000; 0.000],0.000 [0.000; 0.000],0.000 [0.000; 0.000],0.251 [0.234; 0.270]
Persistencia,0.787 [0.774; 0.800],0.588 [0.557; 0.617],0.823 [0.812; 0.834],0.712 [0.687; 0.734],0.697 [0.673; 0.719],0.705 [0.680; 0.727],0.578 [0.551; 0.603],0.177 [0.166; 0.188]
Persistencia (8 sem.),0.859 [0.849; 0.869],0.728 [0.699; 0.754],0.805 [0.794; 0.816],0.655 [0.629; 0.678],0.756 [0.735; 0.775],0.701 [0.678; 0.723],0.561 [0.536; 0.584],0.135 [0.127; 0.142]
Ingenuo estacional,0.540 [0.527; 0.552],0.325 [0.300; 0.352],0.662 [0.642; 0.680],0.399 [0.359; 0.439],0.230 [0.203; 0.258],0.292 [0.260; 0.323],0.096 [0.065; 0.126],0.338 [0.320; 0.358]


In [6]:
comparaciones = []  # Diferencias pareadas con la mejor línea base
for base in ["Persistencia (8 sem.)", "Persistencia"]:  # Líneas base más fuertes
    for nombre_m, met in [("AUC-ROC", roc_auc_score), ("AUC-PR", average_precision_score)]:  # Métricas
        dif, lo, hi, p = mb.bootstrap_diferencia(te[cf.TARGET], p_te, puntajes[base], te["COD_MUN_N"], metrica=met, B=1000)  # Bootstrap pareado
        comparaciones.append({"comparación": f"Regresión logística − {base}", "métrica": nombre_m, "diferencia": dif,  # Resultado
                              "IC 2,5 %": lo, "IC 97,5 %": hi, "p (bootstrap)": p})
pd.DataFrame(comparaciones).round(4)

,comparación,métrica,diferencia,"IC 2,5 %","IC 97,5 %",p (bootstrap)
0,Regresión logística − Persistencia (8 sem.),AUC-ROC,0.0262,0.0225,0.0296,0.0010
1,Regresión logística − Persistencia (8 sem.),AUC-PR,0.0813,0.0745,0.0883,0.0010
2,Regresión logística − Persistencia,AUC-ROC,0.0980,0.0931,0.1036,0.0010
3,Regresión logística − Persistencia,AUC-PR,0.2206,0.2109,0.2290,0.0010


**Interpretación.** La regresión logística supera con claridad a todas las líneas base en las métricas que importan: AUC-ROC ≈ 0,885 y AUC-PR ≈ 0,81, frente a 0,5 / 0,30 del Dummy y 0,86 / 0,73 de la mejor persistencia; las diferencias pareadas tienen intervalos que no contienen el 0. Su F1 y su MCC también son los más altos. Los intervalos (por municipio) son estrechos, de ± 0,01 en AUC-ROC, porque hay 376 municipios y dos años de prueba.

Ojo con la **exactitud**: la persistencia simple acierta casi lo mismo que el modelo (≈ 0,82) y el Dummy, que nunca alerta, ya llega a 0,70. La exactitud no distingue un buen sistema de alerta de uno trivial; por eso no se usa para decidir. El ingenuo estacional es casi inútil (AUC ≈ 0,54): los brotes no se repiten en la misma semana del año, lo que confirma la estacionalidad débil encontrada en el EDA.

## Matriz de confusión, curvas ROC y precisión-recall

In [7]:
pred = (p_te >= umbral).astype(int)  # Alertas
cm = confusion_matrix(te[cf.TARGET], pred)  # [[VN, FP], [FN, VP]]
etiquetas = [[f"VN<br>{cm[0, 0]:,}", f"FP<br>{cm[0, 1]:,}"], [f"FN<br>{cm[1, 0]:,}", f"VP<br>{cm[1, 1]:,}"]]  # Textos
fig = go.Figure(go.Heatmap(z=cm, x=["Predicho: sin brote", "Predicho: brote"], y=["Real: sin brote", "Real: brote"], text=etiquetas,  # Mapa de calor
                           texttemplate="%{text}", colorscale="Blues", showscale=False, textfont=dict(size=16)))
fig.update_layout(title=f"Matriz de confusión en la prueba (umbral {umbral:.3f})", yaxis_autorange="reversed")  # Formato
mostrar(fig, 380)  # Muestra
print(f"Anticipa {cm[1, 1]:,} de {cm[1].sum():,} brotes (recall {cm[1, 1] / cm[1].sum():.1%}); {cm[0, 1]:,} falsas alarmas; "
      f"{cm[:, 1].sum() / te['time_idx'].nunique():.0f} alertas por semana en promedio")  # Lectura operativa

Anticipa 8,453 de 11,395 brotes (recall 74.2%); 3,624 falsas alarmas; 121 alertas por semana en promedio


In [8]:
fig = make_subplots(1, 2, subplot_titles=("Curva ROC", "Curva precisión-recall"))  # Dos paneles
colores = {"Regresión logística": AZUL, "Persistencia (8 sem.)": VERDE, "Persistencia": AMBAR, "Ingenuo estacional": MORADO, "Dummy (prior)": GRIS}  # Colores
for nombre, col in colores.items():  # Cada modelo
    p = puntajes[nombre]  # Puntajes
    fpr, tpr, _ = roc_curve(te[cf.TARGET], p)  # ROC
    pr, rc, _ = precision_recall_curve(te[cf.TARGET], p)  # PR
    fig.add_scatter(x=fpr, y=tpr, name=f"{nombre} ({roc_auc_score(te[cf.TARGET], p):.3f})", line=dict(color=col), row=1, col=1)  # ROC
    fig.add_scatter(x=rc, y=pr, showlegend=False, line=dict(color=col), row=1, col=2)  # PR
fig.add_scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(color=GRIS, dash="dot"), showlegend=False, row=1, col=1)  # Azar ROC
fig.add_hline(y=te[cf.TARGET].mean(), line_dash="dot", line_color=GRIS, row=1, col=2)  # Azar PR = prevalencia
fig.update_xaxes(title_text="tasa de falsos positivos", row=1, col=1)  # Ejes
fig.update_yaxes(title_text="recall", row=1, col=1)  # Ejes
fig.update_xaxes(title_text="recall", row=1, col=2)  # Ejes
fig.update_yaxes(title_text="precisión", row=1, col=2)  # Ejes
fig.update_layout(title="Curvas en la prueba 2023–2024 (leyenda: AUC-ROC)")  # Título
mostrar(fig, 440)  # Muestra

## Calibración

In [9]:
frac_te, media_te, ece_te = mb.calibracion(te[cf.TARGET].to_numpy(), p_te)  # Calibración en prueba
frac_va, media_va, ece_va = mb.calibracion(va[cf.TARGET].to_numpy(), p_va)  # Calibración en validación
fig = go.Figure()  # Curva de confiabilidad
fig.add_scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(color=GRIS, dash="dot"), name="Calibración perfecta")  # Diagonal
fig.add_scatter(x=media_va, y=frac_va, mode="lines+markers", line=dict(color=VERDE), name=f"Validación 2022 (ECE {ece_va:.3f})")  # Validación
fig.add_scatter(x=media_te, y=frac_te, mode="lines+markers", line=dict(color=AZUL), name=f"Prueba 2023–24 (ECE {ece_te:.3f})")  # Prueba
fig.update_layout(title="Curva de calibración por deciles de probabilidad", xaxis_title="probabilidad predicha", yaxis_title="proporción observada de brotes")  # Formato
mostrar(fig, 420)  # Muestra
print(f"Brier: validación {np.mean((p_va - va[cf.TARGET]) ** 2):.4f} · prueba {np.mean((p_te - te[cf.TARGET]) ** 2):.4f} · "
      f"probabilidad media en prueba {p_te.mean():.3f} frente a prevalencia {te[cf.TARGET].mean():.3f}")  # Resumen

Brier: validación 0.0673 · prueba 0.1217 · probabilidad media en prueba 0.257 frente a prevalencia 0.303


**Interpretación.** En la validación de 2022 (prevalencia parecida a la del entrenamiento) el error de calibración es pequeño (ECE ≈ 0,03), aunque el modelo ya subestima un poco (probabilidad media 9,7 % frente a 12,4 % observado). En la prueba la **subestimación** crece (ECE ≈ 0,05; 25,7 % frente a 30,3 %), sobre todo en las probabilidades intermedias: la prevalencia de 2023–2024 (≈ 30 %) triplica la del entrenamiento (≈ 10 %), y la regresión logística aprendió una intersección acorde con la prevalencia antigua. Es la consecuencia directa del *prior shift* documentado en el EDA. El ordenamiento (AUC) se mantiene; lo que se desajusta es la escala, y se corrige recalibrando la intersección o el umbral con datos recientes.

## Diagnóstico de residuos: autocorrelación temporal y espacial

Residuo de respuesta: $r = y - \hat p$. Si el modelo capturara toda la estructura, los residuos no tendrían dependencia temporal (ACF ≈ 0) ni espacial (I de Moran ≈ 0).

In [10]:
resid = (te[cf.TARGET] - p_te).to_numpy()  # Residuos en la prueba
acf_r, n_mun = mb.acf_residuos(te, resid, rezagos=12)  # ACF media dentro de cada municipio
banda = 1.96 / np.sqrt(te.groupby("COD_MUN_N").size().mean())  # Banda aproximada por serie
fig = go.Figure(go.Bar(x=np.arange(1, 13), y=acf_r, marker_color=AZUL))  # ACF
fig.add_hline(y=banda, line_dash="dot", line_color=GRIS)  # Banda
fig.add_hline(y=-banda, line_dash="dot", line_color=GRIS)  # Banda
fig.update_layout(title=f"ACF media de los residuos dentro de los municipios ({n_mun} series)", xaxis_title="rezago (semanas)", yaxis_title="autocorrelación")  # Formato
mostrar(fig, 340)  # Muestra
m = d.municipios()  # Centroides
D = sp.haversine(m["latitud"].to_numpy(), m["longitud"].to_numpy())  # Distancias
r_mun = te.assign(r=resid).groupby("COD_MUN_N")["r"].mean().loc[m["COD_MUN_N"]]  # Residuo medio por municipio
mor = sp.moran(r_mun.to_numpy(), sp.pesos_knn(D, 6), permutaciones=999)  # Moran de los residuos
print(f"ACF de los residuos: rezago 1 = {acf_r[0]:.3f}, rezago 4 = {acf_r[3]:.3f} · I de Moran del residuo medio por municipio: {mor['I']:.3f} (p = {mor['p']:.3f})")  # Resumen
fig = px.scatter_geo(m.assign(residuo=r_mun.to_numpy()), lat="latitud", lon="longitud", color="residuo", hover_name="municipio",  # Mapa de residuos
                     color_continuous_scale="RdBu_r", color_continuous_midpoint=0)
fig.update_traces(marker=dict(size=7, line=dict(width=0.3)))  # Tamaño
fig.update_layout(title="Residuo medio por municipio en la prueba (rojo: brotes no anticipados)")  # Título
mostrar(mapa_base(fig), 540)  # Muestra

ACF de los residuos: rezago 1 = 0.339, rezago 4 = 0.010 · I de Moran del residuo medio por municipio: 0.158 (p = 0.001)


**Interpretación.** Los residuos **sí tienen dependencia**: autocorrelación temporal clara en los primeros 3 rezagos (los episodios de brote duran varias semanas y el modelo se equivoca en rachas) y autocorrelación espacial significativa (municipios vecinos comparten el signo del error, con zonas enteras subestimadas en 2023–2024). Según la guía, esto indica **información estructural sin capturar**: la dinámica de vecindad (Knox y Moran del EDA) y la no linealidad del clima. Es la motivación para los modelos de los siguientes entregables: variables de rezago espacial y modelos no lineales.

## Curva de aprendizaje y sobreajuste

In [11]:
curva = mb.curva_aprendizaje(tr, va, C=grid.best_params_["clf__C"], cw=grid.best_params_["clf__class_weight"])  # Fracciones de municipios
fig = make_subplots(1, 2, subplot_titles=("AUC-ROC", "AUC-PR"))  # Dos paneles
for i, met in enumerate(["AUC-ROC", "AUC-PR"]):  # Métricas
    fig.add_scatter(x=curva["filas"], y=curva[f"{met} entrenamiento"], name="Entrenamiento", line=dict(color=AZUL), showlegend=i == 0, row=1, col=i + 1)  # Entrenamiento
    fig.add_scatter(x=curva["filas"], y=curva[f"{met} validación"], name="Validación 2022", line=dict(color=VERDE), showlegend=i == 0, row=1, col=i + 1)  # Validación
fig.update_xaxes(title_text="filas de entrenamiento")  # Eje
fig.update_layout(title="Curva de aprendizaje (fracciones crecientes de municipios)")  # Título
mostrar(fig, 360)  # Muestra
curva.round(4)

,fraccion,filas,AUC-PR entrenamiento,AUC-PR validación,AUC-ROC entrenamiento,AUC-ROC validación
0,0.1000,13414,0.7160,0.6722,0.9202,0.9028
1,0.2000,26475,0.6758,0.6730,0.9163,0.8977
2,0.3500,46596,0.6513,0.6721,0.9088,0.8982
3,0.5000,66364,0.6356,0.6708,0.9075,0.8998
4,0.7500,99546,0.6417,0.6714,0.9070,0.9005
5,1.0000,132728,0.6532,0.6696,0.9067,0.8978


In [12]:
p_tr = modelo.predict_proba(tr[cf.FEATURES])[:, 1]  # Probabilidades en entrenamiento
brecha = pd.DataFrame({conj: mb.metricas(yy, pp, umbral) for conj, yy, pp in [("Entrenamiento 2015–21", tr[cf.TARGET], p_tr),  # Mismo umbral
                                                                            ("Validación 2022", va[cf.TARGET], p_va), ("Prueba 2023–24", te[cf.TARGET], p_te)]}).T
brecha.round(4)

,AUC-ROC,AUC-PR,Accuracy,Precision,Recall,F1,MCC,Brier
Entrenamiento 2015–21,0.9067,0.6532,0.9178,0.5950,0.6266,0.6104,0.5647,0.0558
Validación 2022,0.8978,0.6696,0.9115,0.6632,0.5751,0.6160,0.5681,0.0673
Prueba 2023–24,0.8855,0.8090,0.8254,0.6999,0.7418,0.7203,0.5940,0.1217


**Interpretación.** La curva de aprendizaje es plana: con el 10 % de los municipios el modelo ya alcanza casi el mismo AUC de validación que con todos, y las curvas de entrenamiento y validación están cerca. El tamaño de muestra es más que suficiente para un modelo lineal y **no hay sobreajuste** (la brecha de AUC-ROC entre entrenamiento y prueba es de unas dos centésimas). El límite es de **sesgo**: el modelo es demasiado simple para la estructura no lineal y espacial; más datos no lo mejorarán, pero un modelo más flexible sí podría. Las diferencias de AUC-PR, precisión y Brier entre conjuntos se deben sobre todo al cambio de prevalencia, no al sobreajuste.

## Interpretación de los coeficientes

Los coeficientes están en unidades de desviación estándar (las variables se estandarizan): $e^{\beta}$ es el cambio en las *odds* de brote cuando la variable sube una desviación estándar, manteniendo las demás fijas. Los intervalos se obtienen reajustando el Pipeline en 200 remuestreos de municipios.

In [13]:
coef = mb.coeficientes_bootstrap(tr, grid.best_params_["clf__C"], grid.best_params_["clf__class_weight"], B=200)  # Coeficientes e IC
coef["odds ratio por 1 DE"] = np.exp(coef["coef"])  # Razón de odds
coef["IC excluye 0"] = (coef["IC 2,5 %"] > 0) | (coef["IC 97,5 %"] < 0)  # Significativo
orden = coef.sort_values("coef")  # Orden
fig = go.Figure(go.Bar(x=orden["coef"], y=orden["variable"], orientation="h", marker_color=np.where(orden["coef"] > 0, ROJO, AZUL),  # Coeficientes
                       error_x=dict(type="data", symmetric=False, array=orden["IC 97,5 %"] - orden["coef"], arrayminus=orden["coef"] - orden["IC 2,5 %"])))
fig.update_layout(title="Coeficientes estandarizados con IC 95 % (bootstrap de municipios)", xaxis_title="coeficiente (log-odds por 1 DE)")  # Formato
mostrar(fig, 760)  # Muestra
coef.set_index("variable").sort_values("coef", key=np.abs, ascending=False).round(3)

,coef,"IC 2,5 %","IC 97,5 %",odds ratio por 1 DE,IC excluye 0
variable,,,,,
brote_m8,0.6020,0.4810,0.6580,1.8260,True
lc_l1,0.4340,0.3720,0.5000,1.5430,True
ratio_umbral,0.4290,0.3810,0.4850,1.5360,True
lc_m12,-0.4020,-0.5330,-0.2310,0.6690,True
lc_m4,0.2580,0.2300,0.2850,1.2940,True
lc_l0,0.2440,0.1810,0.2970,1.2770,True
oni_anom,0.2300,0.1710,0.2870,1.2590,True
ndvi_mean_m12,-0.2040,-0.3070,-0.1160,0.8150,True
lc_l2,0.1970,0.1510,0.2520,1.2180,True


**Interpretación.**

* **Historia de casos.** Los coeficientes más grandes y estables: la proporción reciente de semanas en brote (`brote_m8`), los casos de las últimas semanas (`lc_l0`, `lc_l1`, `lc_m4`) y los casos frente al umbral objetivo (`ratio_umbral`) aumentan el riesgo. Las medias largas (`lc_m8`, `lc_m12`) tienen signo **negativo** condicional a las cortas: con los casos recientes fijos, una historia larga alta significa que el municipio ya venía en epidemia (y el brote tiende a agotarse); lo que predice un brote es la **aceleración** reciente. Es un efecto de colinealidad que debe leerse en conjunto, no variable por variable.
* **El Niño.** El ONI actual aumenta el riesgo (una desviación estándar multiplica las *odds* por ≈ 1,26).
* **Clima local.** Efectos pequeños, de signos mezclados entre ventanas (lluvia de 4–8 semanas positiva, de 12 negativa), coherentes con las relaciones no lineales vistas en el EDA. Los de temperatura no son distinguibles de 0: con el 64 % de la temperatura imputada, su señal está muy atenuada.
* **Contexto.** Más flujo de personas, más riesgo. La estacionalidad y la bandera de temperatura imputada no son distinguibles de 0.

## Generalización a zonas nuevas: esquemas espaciales y espacio-temporales

Bloques por departamento (`GroupKFold`, 5 bloques) con 50 km de separación: se quitan del entrenamiento los municipios a menos de 50 km de cualquier municipio del bloque evaluado.

In [14]:
from sklearn.model_selection import GroupKFold  # Bloques espaciales
mun_xy = m.set_index("COD_MUN_N")[["latitud", "longitud"]]  # Coordenadas por código
filas = []  # Resultados
for k, (i_tr, i_ev) in enumerate(GroupKFold(5).split(tr, groups=tr["depto"])):  # Bloques de departamentos
    deptos = set(tr.iloc[i_ev]["depto"])  # Departamentos evaluados
    m_ev = tr.iloc[i_ev]["COD_MUN_N"].unique()  # Municipios evaluados
    m_tr = tr.iloc[i_tr]["COD_MUN_N"].unique()  # Municipios candidatos al entrenamiento
    Dx = sp.haversine(np.r_[mun_xy.loc[m_tr, "latitud"], mun_xy.loc[m_ev, "latitud"]], np.r_[mun_xy.loc[m_tr, "longitud"], mun_xy.loc[m_ev, "longitud"]])[: len(m_tr), len(m_tr):]  # Distancias cruzadas
    lejos = m_tr[Dx.min(axis=1) > 50]  # Municipios a más de 50 km del bloque evaluado
    ent = tr[tr["COD_MUN_N"].isin(lejos)]  # Entrenamiento con zona de separación
    mdl = mb.pipeline_base().set_params(clf__C=grid.best_params_["clf__C"], clf__class_weight=grid.best_params_["clf__class_weight"]).fit(ent[cf.FEATURES], ent[cf.TARGET])  # Ajuste
    ev_mismo = tr.iloc[i_ev]  # Zonas nuevas, mismo periodo
    ev_futuro = te[te["depto"].isin(deptos)]  # Zonas nuevas y futuro
    filas.append({"bloque": k + 1, "departamentos": len(deptos), "municipios evaluados": len(m_ev), "municipios excluidos por la zona de 50 km": len(m_tr) - len(lejos),
                  "AUC-ROC zonas nuevas (2015–21)": roc_auc_score(ev_mismo[cf.TARGET], mdl.predict_proba(ev_mismo[cf.FEATURES])[:, 1]),
                  "AUC-ROC zonas nuevas + futuro (2023–24)": roc_auc_score(ev_futuro[cf.TARGET], mdl.predict_proba(ev_futuro[cf.FEATURES])[:, 1]),
                  "AUC-PR zonas nuevas + futuro (2023–24)": average_precision_score(ev_futuro[cf.TARGET], mdl.predict_proba(ev_futuro[cf.FEATURES])[:, 1])})
bloques = pd.DataFrame(filas).set_index("bloque")  # Tabla
bloques.round(4)

,departamentos,municipios evaluados,municipios excluidos por la zona de 50 km,AUC-ROC zonas nuevas (2015–21),AUC-ROC zonas nuevas + futuro (2023–24),AUC-PR zonas nuevas + futuro (2023–24)
bloque,,,,,,
1,7,76,32,0.9056,0.8826,0.7735
2,6,75,44,0.9008,0.8639,0.7813
3,6,75,39,0.8987,0.9044,0.8627
4,7,75,52,0.9276,0.8952,0.8125
5,6,75,27,0.8834,0.8695,0.7851


In [15]:
esquemas = pd.DataFrame({  # Resumen de los tres esquemas
    "AUC-ROC": [roc_auc_score(te[cf.TARGET], p_te), bloques["AUC-ROC zonas nuevas (2015–21)"].mean(), bloques["AUC-ROC zonas nuevas + futuro (2023–24)"].mean()],
    "dispersión entre bloques": [np.nan, bloques["AUC-ROC zonas nuevas (2015–21)"].std(), bloques["AUC-ROC zonas nuevas + futuro (2023–24)"].std()],
}, index=["1. Futuro, mismas zonas (prueba 2023–24)", "2. Zonas nuevas, mismo periodo", "3. Zonas nuevas y futuro"])
esquemas.round(4)

,AUC-ROC,dispersión entre bloques
"1. Futuro, mismas zonas (prueba 2023–24)",0.8855,NaN
"2. Zonas nuevas, mismo periodo",0.9032,0.0160
3. Zonas nuevas y futuro,0.8831,0.0170


**Interpretación.** El modelo **generaliza a departamentos que no vio**, incluso con la zona de separación de 50 km: el AUC por bloques es comparable al del esquema principal, con variación entre bloques (algunas regiones son más difíciles). Esto es esperable: las predictoras son relativas a cada municipio (casos frente a su canal endémico, anomalías climáticas) y no identifican lugares. El esquema 3 (zonas nuevas y futuro) es el más exigente y el más variable.

## Nota crítica: ¿un desempeño demasiado alto?

La exactitud en la prueba es ≈ 0,83, dentro del rango 80–90 % que la guía pide revisar. La revisión:

1. **Fuga de datos.** Auditada en el capítulo 6: ninguna predictora supera un AUC univariado de 0,9, la columna con información del futuro se descartó, la partición es temporal con purga y las particiones no comparten filas.
2. **Comparación con la línea base trivial.** El 83 % no es mérito del modelo: el Dummy que nunca alerta acierta el 70 % y la persistencia simple, el 82 %. Por eso el modelo se juzga por AUC-ROC, AUC-PR, F1 y MCC, donde sí supera a las líneas base con intervalos que excluyen el 0.
3. **Estructura temporal y espacial.** La validación respeta el tiempo (años hacia adelante con purga) y el espacio (bloques de departamentos con 50 km de separación); el desempeño se mantiene en ambos esquemas.
4. **Complejidad del dataset.** El problema no es trivial: la mejor regla simple llega a AUC-PR 0,73 y el modelo a 0,81, los residuos conservan estructura espacial y temporal sin explicar y el clima tiene efectos no lineales. Hay margen claro para los modelos de los siguientes entregables.

**Conclusión:** el desempeño alto en exactitud es consecuencia del desbalance y de la persistencia natural de las epidemias, no de una fuga; el dataset es adecuado para el curso.